In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
# Argumentos de estabilidad para Docker
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

try:
    # El Manager instala el driver compatible con la versión de Chrome instalada
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), 
        options=options
    )
    
    driver.get("https://www.google.com")
    print(f"¡CONECTADO! Título de la página: {driver.title}")
    driver.quit()
    
except Exception as e:
    print(f" Error: {e}")

¡CONECTADO! Título de la página: Google


In [9]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

def iniciar_navegador():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

URL_OBJETIVO = "https://www.pisos.com/venta/pisos-madrid/"

SELECTOR_CONTENEDOR = "div.ad-preview__info"
SELECTOR_DATO_A = "p.ad-preview__subtitle"
SELECTOR_DATO_B = "span.ad-preview__price"

driver = iniciar_navegador()
dataset_final = []

try:
    print("Conectando a la fuente de datos...")
    driver.get(URL_OBJETIVO)
    time.sleep(10)

    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
    print(f"Se encontraron {len(elementos)} registros potenciales.")

    for i, item in enumerate(elementos, start=1):
        try:
            ubicacion = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).get_attribute("textContent").strip()
            precio = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).get_attribute("textContent").strip()

            if not ubicacion or not precio:
                continue

            valor_limpio = "".join(filter(str.isdigit, precio))

            print(f"Registro válido {i}: ubicación='{ubicacion}' | precio='{precio}'")

            dataset_final.append({
                "ubicacion": ubicacion,
                "precio": valor_limpio
            })

        except Exception as e:
            print(f"Error en registro {i}: {e}")
            continue

    if dataset_final:
        df = pd.DataFrame(dataset_final)
        df.to_csv("datos_extraidos_PisoMadrid.csv", index=False)
        print("Proceso exitoso. Archivo generado: datos_extraidos_grupo.csv")
        print(df.head())
    else:
        print("No se capturaron datos. Revisar estructura o selectores.")

finally:
    driver.quit()
    print("Navegador cerrado.")

Conectando a la fuente de datos...
Se encontraron 30 registros potenciales.
Registro válido 1: ubicación='Universidad-Malasaña (Distrito Centro. Madrid Capital)' | precio='570.000 €'
Registro válido 2: ubicación='Centro (Rivas-Vaciamadrid)' | precio='699.000 €'
Registro válido 3: ubicación='Sevilla la Nueva' | precio='406.900 €'
Registro válido 4: ubicación='Lucero (Distrito Latina. Madrid Capital)' | precio='259.000 €'
Registro válido 5: ubicación='Gaztambide (Distrito Chamberí. Madrid Capital)' | precio='1.190.000 €'
Registro válido 6: ubicación='Cármenes (Distrito Latina. Madrid Capital)' | precio='550.000 €'
Registro válido 7: ubicación='Guindalera (Distrito Barrio de Salamanca. Madrid Capital)' | precio='390.000 €'
Registro válido 8: ubicación='Palomeras Sureste (Distrito Puente de Vallecas. Madrid Capital)' | precio='369.000 €'
Registro válido 9: ubicación='Casco Histórico de Barajas (Distrito Barajas. Madrid Capital)' | precio='420.000 €'
Registro válido 10: ubicación='Orcasur (